In [ ]:
import os
import sys
from pathlib import Path

# Make `src` importable regardless of notebook working directory.
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.features import encode_matchups_symmetric, compute_efficiency, compute_four_factors
from src.massey import load_massey_features
from src.elo import compute_elo_men, compute_elo_women
from src.paths import get_data_dir

DATA_DIR = Path(get_data_dir())
print("Using DATA_DIR:", str(DATA_DIR))

# Build feature_store if not already present.
if "feature_store" not in globals():
    print("feature_store not found — building from CSVs...")

    # Elo (men + women) into one dict, keyed by (Season, TeamID)
    elo_m = compute_elo_men(str(DATA_DIR))
    elo_w = compute_elo_women(str(DATA_DIR))
    elo_all = dict(elo_m)
    elo_all.update(elo_w)

    # Efficiency / Four Factors (regular season detailed)
    m_det = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
    w_det = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
    efficiency = pd.concat([compute_efficiency(m_det), compute_efficiency(w_det)], ignore_index=True)
    four_factors = pd.concat([compute_four_factors(m_det), compute_four_factors(w_det)], ignore_index=True)

    # Massey consensus (men-only file; used as a generic feature table)
    massey = load_massey_features(str(DATA_DIR), min_coverage=0.8)

    # Seeds
    def _seed_num(seed_str: str) -> int | None:
        if not isinstance(seed_str, str):
            return None
        digits = "".join([c for c in seed_str if c.isdigit()])
        if not digits:
            return None
        return int(digits[:2]) if len(digits) >= 2 else int(digits)

    seeds_m_raw = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")[["Season", "TeamID", "Seed"]].copy()
    seeds_w_raw = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")[["Season", "TeamID", "Seed"]].copy()

    seeds_m = seeds_m_raw.assign(
        Season=seeds_m_raw["Season"].astype(int),
        TeamID=seeds_m_raw["TeamID"].astype(int),
        Seed=seeds_m_raw["Seed"].astype(str).map(_seed_num),
    ).dropna(subset=["Seed"])
    seeds_w = seeds_w_raw.assign(
        Season=seeds_w_raw["Season"].astype(int),
        TeamID=seeds_w_raw["TeamID"].astype(int),
        Seed=seeds_w_raw["Seed"].astype(str).map(_seed_num),
    ).dropna(subset=["Seed"])

    feature_store = {
        "elo": elo_all,
        "efficiency": efficiency,
        "four_factors": four_factors,
        "massey": massey,
        "seeds_m": seeds_m[["Season", "TeamID", "Seed"]].copy(),
        "seeds_w": seeds_w[["Season", "TeamID", "Seed"]].copy(),
    }

    print("feature_store built:", {k: (len(v) if isinstance(v, dict) else v.shape) for k, v in feature_store.items()})
else:
    print("feature_store already exists — reusing it.")

def build_encoded_training_set(
    tourney_compact_df: pd.DataFrame,
    *,
    elo_dict: dict[tuple[int, int], float],
    efficiency_df: pd.DataFrame,
    four_factors_df: pd.DataFrame,
    massey_df: pd.DataFrame,
    seeds_df: pd.DataFrame,
    label_col_name: str = "label",
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, pd.Series]:
    # Build competition-convention rows: Team1ID is the lower TeamID.
    season = tourney_compact_df["Season"].astype(int).to_numpy()
    w_team = tourney_compact_df["WTeamID"].astype(int).to_numpy()
    l_team = tourney_compact_df["LTeamID"].astype(int).to_numpy()

    team1_id = np.minimum(w_team, l_team)
    team2_id = np.maximum(w_team, l_team)

    y = (w_team == team1_id).astype(int)

    matchups_df = pd.DataFrame(
        {"Season": season, "Team1ID": team1_id, "Team2ID": team2_id}
    )
    encoded = encode_matchups_symmetric(
        matchups_df=matchups_df,
        elo_dict=elo_dict,
        efficiency_df=efficiency_df,
        four_factors_df=four_factors_df,
        massey_df=massey_df,
        seeds_df=seeds_df,
        label_col=pd.Series(y, name=label_col_name),
    )

    X = encoded.drop(columns=[label_col_name]).copy()
    y_out = encoded[label_col_name].astype(int).to_numpy()

    # Symmetric encoding doubles rows in original order.
    season_out = pd.Series(np.repeat(season, 2).astype(int), name="Season")

    # Recency sample weights aligned to encoded rows.
    w = np.exp(-0.1 * (2024 - season_out.to_numpy())).astype(float)

    return X, y_out, w, season_out


# Feature store already computed externally (per prompt assumption).
# feature_store keys expected:
#   'elo', 'efficiency', 'four_factors', 'massey', 'seeds_m', 'seeds_w'
assert "feature_store" in globals(), "Expected `feature_store` to exist (per prompt assumption)."

train_seasons = np.arange(2010, 2021)  # 2010-2020 inclusive
val_seasons = np.array([2021, 2022])
test_seasons = np.array([2023, 2024, 2025])

mn_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
wn_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

mn_train = mn_compact[mn_compact["Season"].astype(int).isin(train_seasons)].copy()
wn_train = wn_compact[wn_compact["Season"].astype(int).isin(train_seasons)].copy()

X_train_men, y_train_men, w_train_men, season_train_men = build_encoded_training_set(
    mn_train,
    elo_dict=feature_store["elo"],
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)

X_train_women, y_train_women, w_train_women, season_train_women = build_encoded_training_set(
    wn_train,
    elo_dict=feature_store["elo"],
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)

print("CELL 1 — Training set built (symmetric encoding enabled)")
print("X_train_men shape:", X_train_men.shape)
print("X_train_women shape:", X_train_women.shape)
print("season_train_men shape:", season_train_men.shape)
print("season_train_women shape:", season_train_women.shape)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

# -- Guards -----------------------------------------------------------------
assert "X_train_men" in globals(), "Run CELL 1 first."
assert "y_train_men" in globals(), "Run CELL 1 first."
assert "w_train_men" in globals(), "Run CELL 1 first."
assert "season_train_men" in globals(), "CELL 1 must expose season_train_men."

baseline_cols = ["elo_diff", "seed_diff"]
assert all(c in X_train_men.columns for c in baseline_cols), f"Missing columns: {baseline_cols}"

# -- Time-respecting split ---------------------------------------------------
# Train: seasons < 2019
# Val:   2019-2022
# Keep 2023-2025 untouched for final evaluation.
VAL_SEASONS = [2019, 2020, 2021, 2022]
TRAIN_CUTOFF = min(VAL_SEASONS)

is_val = season_train_men.isin(VAL_SEASONS)
is_train = season_train_men < TRAIN_CUTOFF

X_tr = X_train_men.loc[is_train, baseline_cols]
y_tr = y_train_men[is_train.values]
w_tr = w_train_men[is_train.values]

X_vl = X_train_men.loc[is_val, baseline_cols]
y_vl = y_train_men[is_val.values]
w_vl = w_train_men[is_val.values]

print("CELL 2 — Baseline logistic regression (elo_diff + seed_diff)")
print(f"  Train rows: {len(y_tr):,}  (seasons < {TRAIN_CUTOFF})")
print(f"  Val rows:   {len(y_vl):,}  (seasons {VAL_SEASONS})")

# -- Fit and score -----------------------------------------------------------
baseline_lr_men = LogisticRegression(max_iter=5000)
baseline_lr_men.fit(X_tr, y_tr, sample_weight=w_tr)

preds_val = baseline_lr_men.predict_proba(X_vl)[:, 1]

baseline_brier = brier_score_loss(y_vl, preds_val, sample_weight=w_vl)
naive_brier = 0.25  # predict 0.5 always

print(f"\n  Brier (baseline LR, val):  {baseline_brier:.4f}")
print(f"  Brier (naive 0.5 always):  {naive_brier:.4f}")
print(f"  Improvement over naive:    {naive_brier - baseline_brier:.4f}")

# -- Meaningful sanity check -------------------------------------------------
if baseline_brier >= naive_brier:
    print("\nFATAL: baseline model is no better than predicting 0.5 for every game.")
    print("Check Elo computation and seed parsing before continuing.")
    raise ValueError("Baseline model has no predictive power.")
else:
    print("\n  Sanity check passed — model beats the naive baseline.")

baseline_brier_men = baseline_brier
print(f"\n  baseline_brier_men = {baseline_brier_men:.4f}")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import brier_score_loss

assert "X_train_men" in globals(), "Expected X_train_men from CELL 1."
assert "y_train_men" in globals(), "Expected y_train_men from CELL 1."
assert "w_train_men" in globals(), "Expected w_train_men from CELL 1."
assert "baseline_brier" in globals(), "Expected baseline_brier from CELL 2."

feature_cols = [c for c in X_train_men.columns if c != "Season"]

X = X_train_men[feature_cols].copy()
y = y_train_men
w = w_train_men

def cv_brier_logreg_extended(X: pd.DataFrame, y: np.ndarray, w: np.ndarray, n_splits: int = 5) -> float:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores: list[float] = []

    for tr_idx, te_idx in kf.split(X):
        model = LogisticRegression(max_iter=5000)
        model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=w[tr_idx])
        p = model.predict_proba(X.iloc[te_idx])[:, 1]
        scores.append(brier_score_loss(y[te_idx], p))

    return float(np.mean(scores))

print("CELL 3 — Extended logistic regression (all features)")
extended_brier = cv_brier_logreg_extended(X, y, w, n_splits=5)
delta = extended_brier - baseline_brier

print("5-fold CV Brier score (extended):", extended_brier)
print("Delta vs baseline:", delta)

extended_lr_men = LogisticRegression(max_iter=5000)
extended_lr_men.fit(X, y, sample_weight=w)
print("Extended logistic regression fitted on full training set.")

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import KFold
from sklearn.metrics import brier_score_loss

assert "X_train_men" in globals(), "Expected X_train_men from CELL 1."
assert "y_train_men" in globals(), "Expected y_train_men from CELL 1."
assert "w_train_men" in globals(), "Expected w_train_men from CELL 1."
assert "feature_cols" in globals(), "Expected feature_cols from CELL 3."

X = X_train_men[feature_cols].copy()
y = y_train_men
w = w_train_men

xgb_params = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42,
)

def cv_brier_xgb(X: pd.DataFrame, y: np.ndarray, w: np.ndarray, params: dict, n_splits: int = 5) -> float:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores: list[float] = []

    for tr_idx, te_idx in kf.split(X):
        model = XGBClassifier(**params)
        model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=w[tr_idx])
        p = model.predict_proba(X.iloc[te_idx])[:, 1]
        scores.append(brier_score_loss(y[te_idx], p))

    return float(np.mean(scores))

print("CELL 4 — XGBoost (men)")
xgb_men_cv_brier = cv_brier_xgb(X, y, w, xgb_params, n_splits=5)
print("5-fold CV Brier score (XGBoost men):", xgb_men_cv_brier)

model_men = XGBClassifier(**xgb_params)
model_men.fit(X, y, sample_weight=w)
print("XGBoost (men) fitted on full training set.")

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import KFold
from sklearn.metrics import brier_score_loss

assert "X_train_women" in globals(), "Expected X_train_women from CELL 1."
assert "y_train_women" in globals(), "Expected y_train_women from CELL 1."
assert "w_train_women" in globals(), "Expected w_train_women from CELL 1."
assert "feature_cols" in globals(), "Expected feature_cols from CELL 3."

X = X_train_women[feature_cols].copy()
y = y_train_women
w = w_train_women

xgb_params = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42,
)

def cv_brier_xgb(X: pd.DataFrame, y: np.ndarray, w: np.ndarray, params: dict, n_splits: int = 5) -> float:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores: list[float] = []

    for tr_idx, te_idx in kf.split(X):
        model = XGBClassifier(**params)
        model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=w[tr_idx])
        p = model.predict_proba(X.iloc[te_idx])[:, 1]
        scores.append(brier_score_loss(y[te_idx], p))

    return float(np.mean(scores))

print("CELL 5 — XGBoost (women)")
xgb_women_cv_brier = cv_brier_xgb(X, y, w, xgb_params, n_splits=5)
print("5-fold CV Brier score (XGBoost women):", xgb_women_cv_brier)

model_women = XGBClassifier(**xgb_params)
model_women.fit(X, y, sample_weight=w)
print("XGBoost (women) fitted on full training set.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import brier_score_loss

from src.features import encode_matchups_symmetric
from src.calibration import TournamentCalibrator, compare_calibration_methods
from src.paths import get_data_dir

DATA_DIR = Path(get_data_dir())
print("Using DATA_DIR:", str(DATA_DIR))

assert "feature_store" in globals(), "Expected feature_store in notebook context."
assert "model_men" in globals(), "Expected model_men from CELL 4."
assert "model_women" in globals(), "Expected model_women from CELL 5."
assert "feature_cols" in globals(), "Expected feature_cols from CELL 3."

def build_encoded_set_for_seasons(
    tourney_compact_df: pd.DataFrame,
    seasons: np.ndarray,
    *,
    elo_dict: dict[tuple[int, int], float],
    efficiency_df: pd.DataFrame,
    four_factors_df: pd.DataFrame,
    massey_df: pd.DataFrame,
    seeds_df: pd.DataFrame,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    df = tourney_compact_df[tourney_compact_df["Season"].astype(int).isin(seasons)].copy()

    season = df["Season"].astype(int).to_numpy()
    w_team = df["WTeamID"].astype(int).to_numpy()
    l_team = df["LTeamID"].astype(int).to_numpy()

    team1_id = np.minimum(w_team, l_team)
    team2_id = np.maximum(w_team, l_team)
    y = (w_team == team1_id).astype(int)

    matchups_df = pd.DataFrame(
        {"Season": season, "Team1ID": team1_id, "Team2ID": team2_id}
    )

    encoded = encode_matchups_symmetric(
        matchups_df=matchups_df,
        elo_dict=elo_dict,
        efficiency_df=efficiency_df,
        four_factors_df=four_factors_df,
        massey_df=massey_df,
        seeds_df=seeds_df,
        label_col=pd.Series(y, name="label"),
    )

    X = encoded.drop(columns=["label"]).copy()
    y_out = encoded["label"].astype(int).to_numpy()

    # Symmetric encoding duplicates each matchup row.
    seasons_out = np.repeat(season, 2).astype(int)
    w = np.exp(-0.1 * (2024 - seasons_out)).astype(float)  # optional for calibration

    return X, y_out, w

cal_seasons = np.array([2019, 2020, 2021, 2022])

mn_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
wn_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

X_cal_men, y_cal_men, _ = build_encoded_set_for_seasons(
    mn_compact,
    cal_seasons,
    elo_dict=feature_store["elo"],
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)

X_cal_women, y_cal_women, _ = build_encoded_set_for_seasons(
    wn_compact,
    cal_seasons,
    elo_dict=feature_store["elo"],
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)

X_cal_men_used = X_cal_men[feature_cols].copy()
X_cal_women_used = X_cal_women[feature_cols].copy()

print("CELL 6 — Calibration (2019-2022 tournament games)")
raw_men = model_men.predict_proba(X_cal_men_used)[:, 1]
raw_women = model_women.predict_proba(X_cal_women_used)[:, 1]

calibrator_men = TournamentCalibrator()
calibrator_men.fit_men(raw_men, y_cal_men)

calibrator_women = TournamentCalibrator()
calibrator_women.fit_women(raw_women, y_cal_women)

probs_men_cal = calibrator_men.transform_men(raw_men)
probs_women_cal = calibrator_women.transform_women(raw_women)

print("Men calibration comparison:")
print(compare_calibration_methods(raw_men, y_cal_men).to_string(index=False))

print("\nWomen calibration comparison:")
print(compare_calibration_methods(raw_women, y_cal_women).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
calibrator_men.plot_reliability_curve(
    y_true=y_cal_men,
    raw_scores=raw_men,
    calibrated_scores=probs_men_cal,
    ax=axes[0],
)
axes[0].set_title("Men")

calibrator_women.plot_reliability_curve(
    y_true=y_cal_women,
    raw_scores=raw_women,
    calibrated_scores=probs_women_cal,
    ax=axes[1],
)
axes[1].set_title("Women")

plt.tight_layout()
plt.show()

print("Men Brier raw vs calibrated:",
      float(brier_score_loss(y_cal_men, raw_men)),
      "->",
      float(brier_score_loss(y_cal_men, probs_men_cal)))
print("Women Brier raw vs calibrated:",
      float(brier_score_loss(y_cal_women, raw_women)),
      "->",
      float(brier_score_loss(y_cal_women, probs_women_cal)))

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

from src.features import encode_matchups_symmetric
from src.calibration import TournamentCalibrator
from src.paths import get_data_dir

DATA_DIR = Path(get_data_dir())
print("Using DATA_DIR:", str(DATA_DIR))

assert "feature_store" in globals(), "Expected feature_store in notebook context."
assert "baseline_lr_men" in globals(), "Expected baseline_lr_men from CELL 2."
assert "extended_lr_men" in globals(), "Expected extended_lr_men from CELL 3."
assert "model_men" in globals(), "Expected model_men from CELL 4."
assert "calibrator_men" in globals(), "Expected calibrator_men from CELL 6."
assert "feature_cols" in globals(), "Expected feature_cols from CELL 3."

def build_encoded_set_for_seasons(
    tourney_compact_df: pd.DataFrame,
    seasons: np.ndarray,
    *,
    elo_dict: dict[tuple[int, int], float],
    efficiency_df: pd.DataFrame,
    four_factors_df: pd.DataFrame,
    massey_df: pd.DataFrame,
    seeds_df: pd.DataFrame,
) -> tuple[pd.DataFrame, np.ndarray]:
    df = tourney_compact_df[tourney_compact_df["Season"].astype(int).isin(seasons)].copy()

    season = df["Season"].astype(int).to_numpy()
    w_team = df["WTeamID"].astype(int).to_numpy()
    l_team = df["LTeamID"].astype(int).to_numpy()

    team1_id = np.minimum(w_team, l_team)
    team2_id = np.maximum(w_team, l_team)
    y = (w_team == team1_id).astype(int)

    matchups_df = pd.DataFrame(
        {"Season": season, "Team1ID": team1_id, "Team2ID": team2_id}
    )

    encoded = encode_matchups_symmetric(
        matchups_df=matchups_df,
        elo_dict=elo_dict,
        efficiency_df=efficiency_df,
        four_factors_df=four_factors_df,
        massey_df=massey_df,
        seeds_df=seeds_df,
        label_col=pd.Series(y, name="label"),
    )

    X = encoded.drop(columns=["label"]).copy()
    y_out = encoded["label"].astype(int).to_numpy()
    return X, y_out

# Locked test window (never used for tuning decisions)
test_seasons = np.array([2023, 2024, 2025])
mn_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")

X_val_men, y_val_men = build_encoded_set_for_seasons(
    mn_compact,
    test_seasons,
    elo_dict=feature_store["elo"],
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)

baseline_cols = ["elo_diff", "seed_diff"]

X_val_used = X_val_men[feature_cols].copy()
X_val_baseline = X_val_men[baseline_cols].copy()

print("CELL 7 — Locked test evaluation on 2023-2025 (Men)")

p_baseline = baseline_lr_men.predict_proba(X_val_baseline)[:, 1]
p_extended = extended_lr_men.predict_proba(X_val_used)[:, 1]
p_xgb = model_men.predict_proba(X_val_used)[:, 1]
p_xgb_cal = calibrator_men.transform_men(p_xgb)

scores = [
    ("Baseline LR (uncalibrated)", float(brier_score_loss(y_val_men, p_baseline))),
    ("Extended LR (uncalibrated)", float(brier_score_loss(y_val_men, p_extended))),
    ("XGBoost (uncalibrated)", float(brier_score_loss(y_val_men, p_xgb))),
    ("XGBoost (calibrated)", float(brier_score_loss(y_val_men, p_xgb_cal))),
]

table = pd.DataFrame(scores, columns=["Model", "Brier Score"]).sort_values("Brier Score")
print(table.to_string(index=False))

In [ ]:
import os
from joblib import dump

assert "model_men" in globals(), "Expected model_men from CELL 4."
assert "model_women" in globals(), "Expected model_women from CELL 5."
assert "calibrator_men" in globals(), "Expected calibrator_men from CELL 6."
assert "calibrator_women" in globals(), "Expected calibrator_women from CELL 6."

os.makedirs("models", exist_ok=True)

paths = {
    "model_men": os.path.join("models", "model_men.joblib"),
    "model_women": os.path.join("models", "model_women.joblib"),
    "calibrator_men": os.path.join("models", "calibrator_men.joblib"),
    "calibrator_women": os.path.join("models", "calibrator_women.joblib"),
}

objs = {
    "model_men": model_men,
    "model_women": model_women,
    "calibrator_men": calibrator_men,
    "calibrator_women": calibrator_women,
}

print("CELL 8 — Saving models")
for name, path in paths.items():
    dump(objs[name], path)
    size_bytes = os.path.getsize(path)
    print(f"{name}: {path} ({size_bytes / (1024 * 1024):.3f} MB)")